In [ ]:
# importing needed modules/libraries
!pip install nltk sentence_transformers transformers
import nltk
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import cross_val_score
from sentence_transformers import SentenceTransformer
from transformers import BertModel, BertTokenizer

nltk.download('treebank')
from nltk.corpus import treebank

In [ ]:
# each sentence is a list of (word, pos_tag) tuples
sentences = treebank.tagged_sents()

# first sentence from the treebank corpus
first_sentence = sentences[0]
print(first_sentence)

# separate the words and the corresponding POS tags
words = []
pos_order = []
for word, pos in first_sentence:
    words.append(word)
    pos_order.append(pos)

#printing sentence and order of POS tags -->  visualization of how the data looks like
print("Sentence:", " ".join(words))
print("POS Order:", " ".join(pos_order))

In [ ]:
#creating (words, labels) pairs from the treebank
data = []
for sentence in sentences:
    tokens, labels = [], []
    for word, pos in sentence:
        tokens.append(word)
        # logic behind this: if the POS tag starts with "V" (any verb) then label as 1, else 0.
        label = 1 if pos.startswith("V") else 0
        labels.append(label)
    data.append((tokens, labels))

#"data" is a list of (token_list, label_list) tuples.

# printing to visualize structure, and to check if something is off
max_matches = 5  # define how many verb matches to to print
match_count = 0

for tokens, labels in data[:5]:
    for word, label in zip(tokens, labels):
        if label == 1:  # check if the word is classified as a verb
            print(f'"{word}" : "verb"')
            match_count += 1
            if match_count >= max_matches:
                break
    if match_count >= max_matches:
        break

In [ ]:
#load the pretained BERT model and tokenizer
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

def get_bert_sentence_embedding(sentence_tokens):
    """
    Given a list of tokens forming a sentence, return the BERT-based sentence embedding.
    This implementation uses the [CLS] token's representation as the sentence embedding.
    """
    #convert token list to a single string
    sentence = " ".join(sentence_tokens)
    #tokenize sentence with appropriate padding and truncation
    inputs = bert_tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
    #disable gradients, only doing inference --> no training.
    with torch.no_grad():
        outputs = bert_model(**inputs)
    #outputs.last_hidden_state has shape (batch_size, sequence_length, hidden_size)
    #embedding for the [CLS] token is at position 0 of the sequence.
    cls_embedding = outputs.last_hidden_state[:, 0, :]
    
    return cls_embedding.numpy()

In [ ]:
def train_logistic_regression(features, labels):
    """
    Train lr model on token features.
    
    Parameters:
      features (np.ndarray): Feature matrix of shape (num_samples, num_features).
      labels (np.ndarray): Binary labels.
      
    Returns:
      clf: Trained logistic regression classifier.
      metrics: Dictionary containing evaluation metrics.
    """
    # splitting into training and test sets.
    X_train, X_test, y_train, y_test = train_test_split(
        features, labels, test_size=0.2, random_state=42
    )
    
    clf = LogisticRegression(max_iter=1000)
    clf.fit(X_train, y_train)
    
    # predictions on the test set.
    predictions = clf.predict(X_test)
    
    # evaluation
    metrics = {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1_score": f1_score(y_test, predictions)
    }
    
    return clf, metrics


if __name__ == "__main__":
    X_text = []
    y = []
    for tokens, labels in data:
        X_text.extend(tokens)    # flatten the token lists into one list of words
        y.extend(labels)         # flatten the labels accordingly
        # LR expects data into a format where each row represents a single sample (an individual word) with its corresponding features


    model = SentenceTransformer('all-MiniLM-L6-v2')
    X_features = model.encode(X_text)

    # Train logistic regression on the Treebank data features
    clf, eval_metrics = train_logistic_regression(X_features, np.array(y))
    print("Evaluation Metrics on actual Treebank data:")
    print(eval_metrics)


In [ ]:
# cross-validation to evaluate model performance using Treebank data features
def cross_validate_model(features, labels):
    clf = LogisticRegression(max_iter=1000)
    accuracy_scores = cross_val_score(clf, features, labels, cv=5, scoring='accuracy')  # 5-fold cross-validation
    print("Cross-validated Accuracy: ", accuracy_scores.mean())

if __name__ == "__main__":
    cross_validate_model(X_features, np.array(y))